In [1]:
pip install torch torchvision opencv-python numpy matplotlib tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import gc
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm
import kagglehub 

# Ensure OpenSlide or TiffFile is available for multi-resolution slide loading
try:
    import openslide
except ImportError:
    os.system('pip install openslide-python tiffslide')
    import openslide

# =====================================================================
# 1. CONFIGURATION
# =====================================================================
# Automatically download and mount the dataset at runtime
print("[+] Fetching PANDA dataset via kagglehub...")
path = kagglehub.competition_download('prostate-cancer-grade-assessment')
print(f"[+] Data ready at: {path}")

class Config:
    DATA_DIR = path
    CSV_PATH = os.path.join(DATA_DIR, "train.csv")
    IMAGE_DIR = os.path.join(DATA_DIR, "train_images")
    
    # Tile Extraction Settings
    TILE_SIZE = 256
    N_TILES = 16  
    NUM_CLASSES = 6
    
    # Training Parameters
    BATCH_SIZE = 16
    EPOCHS = 4
    LR = 3e-4
    NUM_WORKERS = 2
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    SEED = 42

# (The rest of the script remains exactly the same)
# Set random seed
torch.manual_seed(Config.SEED)
np.random.seed(Config.SEED)

# =====================================================================
# 2. TRAIN / VAL / TEST STRATIFIED SPLIT
# =====================================================================
def create_dataset_splits(csv_path: str):
    """
    Splits train.csv into 70% Train, 15% Validation, and 15% Test sets
    stratified by ISUP grade.
    """
    print("[+] Loading PANDA Metadata & Creating Dataset Splits...")
    df = pd.read_csv(csv_path)
    
    # Filter out entries where image files might be missing or corrupted
    df['image_path'] = df['image_id'].apply(lambda x: os.path.join(Config.IMAGE_DIR, f"{x}.tiff"))
    df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
    
    # 70% Train, 30% Temp (Val + Test)
    train_df, temp_df = train_test_split(
        df, test_size=0.30, stratify=df['isup_grade'], random_state=Config.SEED
    )
    
    # Split Temp into 50% Val (15% total) and 50% Test (15% total)
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df['isup_grade'], random_state=Config.SEED
    )
    
    print(f"    - Train Set:      {len(train_df)} slides")
    print(f"    - Validation Set: {len(val_df)} slides")
    print(f"    - Test Set:       {len(test_df)} slides\n")
    
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

# =====================================================================
# 3. TISSUE TILE EXTRACTION & GRID STITCHING
# =====================================================================
def extract_top_tissue_tiles(slide_path: str, tile_size: int = 256, n_tiles: int = 16):
    """
    Opens WSI, tiles the tissue core into equal patches, selects the top N 
    patches with the highest tissue density, and stitches them into a grid.
    """
    slide = openslide.OpenSlide(slide_path)
    
    # Read slide at medium resolution level (level 1 if available, else 0)
    level = 1 if slide.level_count > 1 else 0
    img = np.array(slide.read_region((0, 0), level, slide.level_dimensions[level]).convert('RGB'))
    
    h, w, c = img.shape
    pad_h = (tile_size - h % tile_size) % tile_size
    pad_w = (tile_size - w % tile_size) % tile_size
    
    if pad_h > 0 or pad_w > 0:
        img = np.pad(img, [[0, pad_h], [0, pad_w], [0, 0]], constant_values=255)
        
    h_tiles = img.shape[0] // tile_size
    w_tiles = img.shape[1] // tile_size
    
    # Reshape image into individual patches
    tiles = img.reshape(h_tiles, tile_size, w_tiles, tile_size, c)
    tiles = tiles.transpose(0, 2, 1, 3, 4).reshape(-1, tile_size, tile_size, c)
    
    # Calculate tissue presence (lower sum = darker region = more tissue, less white space)
    sums = tiles.sum(axis=(1, 2, 3))
    selected_indices = np.argsort(sums)[:n_tiles]
    
    # Pad with blank tiles if slide has fewer than N tissue regions
    if len(selected_indices) < n_tiles:
        padded_tiles = np.full((n_tiles, tile_size, tile_size, c), 255, dtype=np.uint8)
        padded_tiles[:len(selected_indices)] = tiles[selected_indices]
        selected_tiles = padded_tiles
    else:
        selected_tiles = tiles[selected_indices]
        
    # Stitch selected tiles into a square grid (e.g., 4x4 grid for N=16)
    grid_dim = int(np.sqrt(n_tiles))
    rows = [np.concatenate(selected_tiles[i*grid_dim:(i+1)*grid_dim], axis=1) for i in range(grid_dim)]
    stitched_grid = np.concatenate(rows, axis=0)
    
    return stitched_grid

# =====================================================================
# 4. PYTORCH DATASET
# =====================================================================
class PANDADataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row['image_path']
        label = row['isup_grade']
        
        # Tile slide on-the-fly
        grid_image = extract_top_tissue_tiles(image_path, tile_size=Config.TILE_SIZE, n_tiles=Config.N_TILES)
        
        if self.transform:
            grid_image = self.transform(grid_image)
            
        return grid_image, torch.tensor(label, dtype=torch.long)

# Transforms
train_transform = T.Compose([
    T.ToTensor(),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# =====================================================================
# 5. MODEL ARCHITECTURE & TRAINER
# =====================================================================
def build_model(num_classes: int = 6):
    """Backbone adapted for multi-class ISUP grading."""
    model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    for images, labels in tqdm(dataloader, desc="Training"):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
    epoch_loss = running_loss / len(dataloader.dataset)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    return epoch_loss, qwk

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    epoch_loss = running_loss / len(dataloader.dataset)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    return epoch_loss, acc, qwk, all_preds, all_labels

# =====================================================================
# 6. MAIN EXECUTION PIPELINE
# =====================================================================
def main():
    # Load Splits
    train_df, val_df, test_df = create_dataset_splits(Config.CSV_PATH)
    
    # Create Datasets & DataLoaders
    train_dataset = PANDADataset(train_df, transform=train_transform)
    val_dataset = PANDADataset(val_df, transform=val_test_transform)
    test_dataset = PANDADataset(test_df, transform=val_test_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True)
    
    # Setup Model, Loss, Optimizer
    model = build_model(Config.NUM_CLASSES).to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler()
    
    best_val_qwk = -1.0
    
    # Training Loop
    print("[+] Starting Model Training...")
    for epoch in range(1, Config.EPOCHS + 1):
        train_loss, train_qwk = train_one_epoch(model, train_loader, criterion, optimizer, scaler, Config.DEVICE)
        val_loss, val_acc, val_qwk, _, _ = evaluate(model, val_loader, criterion, Config.DEVICE)
        
        print(f"Epoch {epoch:02d}/{Config.EPOCHS:02d} | "
              f"Train Loss: {train_loss:.4f} - QWK: {train_qwk:.4f} | "
              f"Val Loss: {val_loss:.4f} - Acc: {val_acc:.4f} - QWK: {val_qwk:.4f}")
        
        # Save Best Model Weights
        if val_qwk > best_val_qwk:
            best_val_qwk = val_qwk
            torch.save(model.state_dict(), "best_panda_resnet34.pth")
            print("    --> Best model saved!")
            
    # Final Evaluation on Held-Out Test Set
    print("\n[+] Running Final Evaluation on Held-Out TEST SET...")
    model.load_state_dict(torch.load("best_panda_resnet34.pth"))
    test_loss, test_acc, test_qwk, test_preds, test_labels = evaluate(model, test_loader, criterion, Config.DEVICE)
    
    print("\n================ TEST SET METRICS ================")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test QWK Score: {test_qwk:.4f}")
    print("\nClassification Report:")
    print(classification_report(test_labels, test_preds, target_names=[f"ISUP_{i}" for i in range(6)]))
    print("Confusion Matrix:")
    print(confusion_matrix(test_labels, test_preds))

if __name__ == "__main__":
    main()

[+] Fetching PANDA dataset via kagglehub...
[+] Data ready at: /kaggle/input/competitions/prostate-cancer-grade-assessment
[+] Loading PANDA Metadata & Creating Dataset Splits...
    - Train Set:      7431 slides
    - Validation Set: 1592 slides
    - Test Set:       1593 slides

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 168MB/s]
/tmp/ipykernel_23/2798069094.py:246: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


[+] Starting Model Training...


Training:   0%|          | 0/465 [00:00<?, ?it/s]/tmp/ipykernel_23/2798069094.py:188: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Evaluating: 100%|██████████| 100/100 [31:24<00:00, 18.84s/it]


Epoch 01/04 | Train Loss: 1.2776 - QWK: 0.6205 | Val Loss: 1.6439 - Acc: 0.4422 - QWK: 0.5469
    --> Best model saved!


...

Epoch 03/04 | Train Loss: 1.0871 - QWK: 0.7136 | Val Loss: 1.0205 - Acc: 0.5923 - QWK: 0.7493
    --> Best model saved!


Training:   0%|          | 0/465 [00:00<?, ?it/s]/tmp/ipykernel_23/2798069094.py:188: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Evaluating: 100%|██████████| 100/100 [26:47<00:00, 16.07s/it]


Epoch 04/04 | Train Loss: 1.0340 - QWK: 0.7505 | Val Loss: 0.9592 - Acc: 0.6281 - QWK: 0.7761
    --> Best model saved!

[+] Running Final Evaluation on Held-Out TEST SET...


Evaluating: 100%|██████████| 100/100 [31:12<00:00, 18.72s/it]


================ TEST SET METRICS ================
Test Loss: 0.9643
Test Accuracy: 0.6309
Test QWK Score: 0.7670

Classification Report:
              precision    recall  f1-score   support

      ISUP_0       0.72      0.97      0.82       434
      ISUP_1       0.72      0.64      0.68       400
      ISUP_2       0.49      0.50      0.50       202
      ISUP_3       0.46      0.33      0.39       186
      ISUP_4       0.60      0.25      0.35       188
      ISUP_5       0.51      0.66      0.57       183

    accuracy                           0.63      1593
   macro avg       0.58      0.56      0.55      1593
weighted avg       0.62      0.63      0.61      1593

Confusion Matrix:
[[419  13   0   0   0   2]
 [ 78 255  53   8   0   6]
 [ 20  56 101  19   4   2]
 [ 19  15  35  62  16  39]
 [ 26  13   9  25  47  68]
 [ 20   2   7  22  11 121]]
